# Data Imports

In [1]:
import pandas as pd
import numpy as np
import json
import math

In [2]:
# import Artists dataset from Getty, csv file downloaded 10/22/26
Artists = pd.read_csv("Artists.csv")
Artists.head(5)

,x,name,bio,nationality,type
0,http://vocab.getty.edu/ulan/500771934,"Aa, Andries van der","North Netherlandish painter, 1743-1776",North Netherlandish,painters (artists)
1,http://vocab.getty.edu/ulan/500557119,"Aabdaoui, Latif","Moroccan graphic artist, 1958-",Moroccan,graphic artists
2,http://vocab.getty.edu/ulan/500632715,"Aa, Ben","Dutch painter, born 1951",Dutch (culture or style),painters (artists)
3,http://vocab.getty.edu/ulan/500546545,"Aaberg, Gunhild","Danish ceramicist, 1939-",Danish (culture or style),ceramicists
4,http://vocab.getty.edu/ulan/500108834,"Aabergh, Gösta","Norwegian architect, active mid-late 20th century",Norwegian (culture),architects


In [3]:
# Import Art Institute of Chicago dataset, jsonl file from 10/22/26
Chicago = pd.read_json("AIofChicago.jsonl", lines = True) 
Chicago.head(5)

,id,title,main_reference_number,department_title,artist_title
0,4,Priest and Boy,1880.1,Prints and Drawings,Lawrence Carmichael Earle
1,9,"Interior of St. Mark's, Venice",1887.232,Arts of the Americas,David Dalhoff Neal
2,11,Self-Portrait,1887.234,Arts of the Americas,Walter Shirlaw
3,16,The Fall of the Giants,1887.249,Prints and Drawings,Salvator Rosa
4,19,"View of Ponte Lugano on the Anio, from Views o...",1887.252,Prints and Drawings,Giovanni Battista Piranesi


In [4]:
# Import The Met's dataset from their github
Met = pd.read_csv("https://github.com/metmuseum/openaccess/raw/refs/heads/master/MetObjects.csv") 

/var/folders/hf/pbt5699545q4bz8978zvx3lw0000gn/T/ipykernel_2452/3515133808.py:2: DtypeWarning: Columns (5,7,10,11,12,13,14,15,34,35,36,37,38,39,40,41,42,43,44,45,46) have mixed types. Specify dtype option on import or set low_memory=False.
  Met = pd.read_csv("https://github.com/metmuseum/openaccess/raw/refs/heads/master/MetObjects.csv")


The columns warned as having mixed types are as follows:

In [5]:
pd.DataFrame({"type": Met.dtypes, "colnum": np.arange(0, len(Met.columns))}).reset_index().loc[[5,7,10,11,12,13,14,15,34,35,36,37,38,39,40,41,42,43,44,45,46]]

,index,type,colnum
5,Gallery Number,object,5
7,AccessionYear,object,7
10,Culture,object,10
11,Period,object,11
12,Dynasty,object,12
13,Reign,object,13
14,Portfolio,object,14
15,Constituent ID,object,15
34,Geography Type,object,34
35,City,object,35


We will focus on a subset of the Met dataset, with variables "Object Number", "Department", "Accession Year", "Object Name", "Title", and "Artist Display Name" chosen to match the Art Institute of Chicago dataset. Because of this, "Accession Year" is the only variable we have to check data type in. 

In [6]:
Met = Met[["Object Number", "Department", "AccessionYear", "Object Name", "Title", "Artist Display Name"]]
Met.head(5)

,Object Number,Department,AccessionYear,Object Name,Title,Artist Display Name
0,1979.486.1,The American Wing,1979.0,Coin,One-dollar Liberty Head Coin,James Barton Longacre
1,1980.264.5,The American Wing,1980.0,Coin,Ten-dollar Liberty Head Coin,Christian Gobrecht
2,67.265.9,The American Wing,1967.0,Coin,Two-and-a-Half Dollar Coin,NaN
3,67.265.10,The American Wing,1967.0,Coin,Two-and-a-Half Dollar Coin,NaN
4,67.265.11,The American Wing,1967.0,Coin,Two-and-a-Half Dollar Coin,NaN


In [7]:
# inspect AccessionYear
Met = Met.astype({"AccessionYear": "str"})
Met["AccessionYear"].unique()

array(['1979.0', '1980.0', '1967.0', '1916.0', '1909.0', '1964.0',
       '1970.0', '1904.0', '1976.0', '1938.0', '1918.0', '1946.0',
       '1960.0', '1910.0', '1922.0', '1923.0', '1930.0', '1934.0',
       '1936.0', '1937.0', '1939.0', '1941.0', '1947.0', '1948.0',
       '1949.0', '1971.0', '1984.0', '1989.0', '1985.0', '1952.0',
       '1968.0', '1919.0', 'nan', '1978.0', '1972.0', '1924.0', '1983.0',
       '1950.0', '1925.0', '1926.0', '1931.0', '1932.0', '1951.0',
       '1953.0', '2006.0', '1969.0', '1975.0', '2025.0', '1981.0',
       '1986.0', '1959.0', '1944.0', '1982.0', '1945.0', '1888.0',
       '1911.0', '1957.0', '1943.0', '1935.0', '1917.0', '1973.0',
       '1940.0', '1913.0', '1962.0', '1914.0', '1966.0', '1954.0',
       '1920.0', '1933.0', '2014.0', '1928.0', '1901.0', '1958.0',
       '1965.0', '1883.0', '1902.0', '1987.0', '1974.0', '1977.0',
       '1915.0', '1942.0', '1912.0', '1927.0', '2023.0', '1896.0',
       '1892.0', '1877.0', '1906.0', '1955.0', '1961.0'

In [8]:
# the beginning of Object Number can be used to find corrected AccessionYear
Met[Met["AccessionYear"] == "2028"]

,Object Number,Department,AccessionYear,Object Name,Title,Artist Display Name
489233,2024.626.7,Photographs,2028,Photograph,Zuma #21,John Divola


In [9]:
# Not all Object Numbers have the full Accession Year as the first split
Met["Object Number"].str.split(".", n = 1, expand = True)[0].head()

0    1979
1    1980
2      67
3      67
4      67
Name: 0, dtype: object

Within Accession Year we see some double values (ex: 1979.0), some integer values (ex: 1941), and some date values (ex: 1973-05-08). We also see some values that are not possible, since they are in the future, such as 2028. When we check the corresponding Object Number, we can see that this piece was likely aquired in 2024. Using just the Accession Number is also not a solution, since some values leave out the century and just have the year (ex: 1967 as 67).

In [10]:
# create new column with cleaned Acession Year as a numeric value
Met["Cleaned_AY"] = Met["AccessionYear"].str.split(".", n = 1, expand = True)[0].str.split("-", n = 1, expand = True)[0]
Met = Met.astype({"Cleaned_AY": "float64"})
Met.head()

,Object Number,Department,AccessionYear,Object Name,Title,Artist Display Name,Cleaned_AY
0,1979.486.1,The American Wing,1979.0,Coin,One-dollar Liberty Head Coin,James Barton Longacre,1979.0
1,1980.264.5,The American Wing,1980.0,Coin,Ten-dollar Liberty Head Coin,Christian Gobrecht,1980.0
2,67.265.9,The American Wing,1967.0,Coin,Two-and-a-Half Dollar Coin,NaN,1967.0
3,67.265.10,The American Wing,1967.0,Coin,Two-and-a-Half Dollar Coin,NaN,1967.0
4,67.265.11,The American Wing,1967.0,Coin,Two-and-a-Half Dollar Coin,NaN,1967.0


In [11]:
# we find 223 instances of an Accession Year in the future
Met[Met["Cleaned_AY"] > 2026]

,Object Number,Department,AccessionYear,Object Name,Title,Artist Display Name,Cleaned_AY
489232,2024.626.6,Photographs,2027,Photograph,Zuma #20,John Divola,2027.0
489233,2024.626.7,Photographs,2028,Photograph,Zuma #21,John Divola,2028.0
489234,2024.626.8a–c,Photographs,2029,Photograph,"51st Street, 18th June 2010, 1.28.45 pm",Paul Graham,2029.0
489235,2024.626.9,Photographs,2030,Photograph,"The Polo Grounds, New York",Irving Penn,2030.0
489236,2024.626.10,Photographs,2031,Photograph,"30,000 kVA Steam Turbine [Power Station] Ruhrc...",Albert Renger-Patzsch,2031.0
...,...,...,...,...,...,...,...
490742,2024.626.224,Photographs,2056,Photograph,New York,Garry Winogrand,2056.0
490743,2024.626.225,Photographs,2056,Photograph,"Salinas, Kansas",Garry Winogrand,2056.0
490744,2024.626.226,Photographs,2056,Photograph,"Candlestick Park, San Francisco",Garry Winogrand,2056.0
490745,2024.626.227,Photographs,2056,Photograph,London,Garry Winogrand,2056.0


In [12]:
# Add Accession Year from Object Number column - INCOMPLETE
Met["AYfromON"] = Met["Object Number"].str.split(".", n = 1, expand = True)[0]
Met.head()

,Object Number,Department,AccessionYear,Object Name,Title,Artist Display Name,Cleaned_AY,AYfromON
0,1979.486.1,The American Wing,1979.0,Coin,One-dollar Liberty Head Coin,James Barton Longacre,1979.0,1979
1,1980.264.5,The American Wing,1980.0,Coin,Ten-dollar Liberty Head Coin,Christian Gobrecht,1980.0,1980
2,67.265.9,The American Wing,1967.0,Coin,Two-and-a-Half Dollar Coin,NaN,1967.0,67
3,67.265.10,The American Wing,1967.0,Coin,Two-and-a-Half Dollar Coin,NaN,1967.0,67
4,67.265.11,The American Wing,1967.0,Coin,Two-and-a-Half Dollar Coin,NaN,1967.0,67


___

In [13]:
# KEEP
Met["Cleaned_AY"].case_when(
    caselist = [
        (Met["Cleaned_AY"] > 2026, Met["AYfromON"])#, # when year is greater than 2026, replace with year from Object Number
        #(Met["Cleaned_AY"])
    ]
)

0         1979.0
1         1980.0
2         1967.0
3         1967.0
4         1967.0
           ...  
503373    1959.0
503374    1959.0
503375    1959.0
503376    1959.0
503377    1959.0
Name: Cleaned_AY, Length: 503378, dtype: object

In [14]:
Met[Met["Object Number"].str.split(".", n = 1, expand = True)[0].str.isnumeric() == False]["AYfromON"].unique()

array(['Inst', 'X', 'I', ..., 'NK5430 ', 'NB1800 ', 'NK1657 '],
      shape=(5218,), dtype=object)

In [15]:
Met["AYfromON"] = Met["AYfromON"].case_when(
    caselist = [
        (Met["Object Number"].str.split(".", expand = True)[0].str.isnumeric() == False, Met["Object Number"].str.split(".", expand = True)[1]),
        (Met["Object Number"].str.split(".", expand = True)[1].str.isnumeric() == False, Met["Object Number"].str.split(".", expand = True)[2]),
        (Met["Object Number"].str.split(".", expand = True)[1].str.isnumeric() == False, Met["Object Number"].str.split(".", expand = True)[2]),
    ]
)

In [16]:
Met["Object Number"].str.findall("(\\d+)")

0            [1979, 486, 1]
1            [1980, 264, 5]
2              [67, 265, 9]
3             [67, 265, 10]
4             [67, 265, 11]
                ...        
503373    [59, 654, 21, 45]
503374    [59, 654, 21, 44]
503375    [59, 654, 21, 48]
503376    [59, 654, 21, 37]
503377    [59, 654, 21, 43]
Name: Object Number, Length: 503378, dtype: object

In [17]:
Met["Object Number"].str.split(".", expand = True)

#.str.isnumeric() == False

,0,1,2,3,4,5,6,7,8,9,10,11
0,1979,486,1,None,None,None,None,None,None,None,None,None
1,1980,264,5,None,None,None,None,None,None,None,None,None
2,67,265,9,None,None,None,None,None,None,None,None,None
3,67,265,10,None,None,None,None,None,None,None,None,None
4,67,265,11,None,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...
503373,59,654,21(45),None,None,None,None,None,None,None,None,None
503374,59,654,21(44),None,None,None,None,None,None,None,None,None
503375,59,654,21(48),None,None,None,None,None,None,None,None,None
503376,59,654,21(37),None,None,None,None,None,None,None,None,None


In [18]:
Met[np.isnan(Met["Cleaned_AY"])]

,Object Number,Department,AccessionYear,Object Name,Title,Artist Display Name,Cleaned_AY,AYfromON
123,Inst.68.4.1,The American Wing,nan,Architectural elements,"Architectural elements from Halsted House, Rye...",NaN,NaN,68
130,Inst.69.8.1,The American Wing,nan,Lamp shade,Lamp Shade,NaN,NaN,69
131,Inst.69.8.2,The American Wing,nan,Lamp shade,Lamp Shade,NaN,NaN,69
237,Inst.X.6.1,The American Wing,nan,Furniture hardware,Furniture Hardware,NaN,NaN,X
390,Inst.1983.2,The American Wing,nan,Bell pull,Bell Pull,NaN,NaN,1983
...,...,...,...,...,...,...,...,...
503316,Ref.1986.1136.50,Drawings and Prints,nan,Valentine,"Lace with cupid and ""curtain""",George Kershaw,NaN,1986
503317,Ref.1986.1136.51,Drawings and Prints,nan,Valentine,Lace Valentine with Cupid and scraps,George Kershaw,NaN,1986
503318,Ref.1986.1136.52,Drawings and Prints,nan,Valentine,Lace Valentine collage in style of Esther Howland,"Anonymous, British, 19th century",NaN,1986
503319,Ref.1986.1136.53,Drawings and Prints,nan,Valentine,Lace Valentine with doors that open,George Kershaw,NaN,1986


## Record Linkage

In [19]:
Chicago.head()

,id,title,main_reference_number,department_title,artist_title
0,4,Priest and Boy,1880.1,Prints and Drawings,Lawrence Carmichael Earle
1,9,"Interior of St. Mark's, Venice",1887.232,Arts of the Americas,David Dalhoff Neal
2,11,Self-Portrait,1887.234,Arts of the Americas,Walter Shirlaw
3,16,The Fall of the Giants,1887.249,Prints and Drawings,Salvator Rosa
4,19,"View of Ponte Lugano on the Anio, from Views o...",1887.252,Prints and Drawings,Giovanni Battista Piranesi


In [21]:
Met.head()

,Object Number,Department,AccessionYear,Object Name,Title,Artist Display Name,Cleaned_AY,AYfromON
0,1979.486.1,The American Wing,1979.0,Coin,One-dollar Liberty Head Coin,James Barton Longacre,1979.0,1979
1,1980.264.5,The American Wing,1980.0,Coin,Ten-dollar Liberty Head Coin,Christian Gobrecht,1980.0,1980
2,67.265.9,The American Wing,1967.0,Coin,Two-and-a-Half Dollar Coin,NaN,1967.0,67
3,67.265.10,The American Wing,1967.0,Coin,Two-and-a-Half Dollar Coin,NaN,1967.0,67
4,67.265.11,The American Wing,1967.0,Coin,Two-and-a-Half Dollar Coin,NaN,1967.0,67


In [22]:
Artists.head()

,x,name,bio,nationality,type
0,http://vocab.getty.edu/ulan/500771934,"Aa, Andries van der","North Netherlandish painter, 1743-1776",North Netherlandish,painters (artists)
1,http://vocab.getty.edu/ulan/500557119,"Aabdaoui, Latif","Moroccan graphic artist, 1958-",Moroccan,graphic artists
2,http://vocab.getty.edu/ulan/500632715,"Aa, Ben","Dutch painter, born 1951",Dutch (culture or style),painters (artists)
3,http://vocab.getty.edu/ulan/500546545,"Aaberg, Gunhild","Danish ceramicist, 1939-",Danish (culture or style),ceramicists
4,http://vocab.getty.edu/ulan/500108834,"Aabergh, Gösta","Norwegian architect, active mid-late 20th century",Norwegian (culture),architects


In [20]:
import recordlinkage as rl

In [21]:
# preprocessing

# standardize artist names across all three datasets
Artists["clean_name"] = (Artists["name"].str.replace(r"^([^,]+),\s*(.+)$", r"\2 \1", regex=True).str.lower().str.strip())
Chicago["clean_artist"] = Chicago["artist_title"].str.lower().str.strip()
Met["clean_artist"] = Met["Artist Display Name"].str.lower().str.strip()

artists_clean = (
    Artists[["x", "name", "clean_name", "bio", "nationality", "type"]]
    .dropna(subset=["clean_name"])
    .drop_duplicates(subset=["clean_name"])
    .reset_index(drop=True)
)

chicago_artists = (
    Chicago[["clean_artist"]]
    .dropna()
    .drop_duplicates()
    .reset_index(drop=True)
)

met_artists = (
    Met[["clean_artist"]]
    .dropna()
    .drop_duplicates()
    .reset_index(drop=True)
)

artists_clean["block_key"] = artists_clean["clean_name"].str[:3]
chicago_artists["block_key"] = chicago_artists["clean_artist"].str[:3]
met_artists["block_key"] = met_artists["clean_artist"].str[:3]

In [22]:
def link_datasets(museum_df, getty_df, threshold=0.85):
    indexer = rl.Index()
    indexer.block("block_key")
    pairs = indexer.index(museum_df, getty_df)
    
    compare = rl.Compare()
    compare.string("clean_artist", "clean_name", method="levenshtein", threshold=threshold, label="name_sim")
    features = compare.compute(pairs, museum_df, getty_df)
    
    candidates = features[features.sum(axis=1) == 1]
    return candidates

In [23]:
def get_matched_pairs(candidates, museum_df, getty_df):
    match_pairs = candidates.index.to_frame(index=False)
    match_pairs.columns = ["museum_index", "getty_index"]
    
    merged = (
        match_pairs
        .merge(museum_df, left_on="museum_index", right_index=True)
        .merge(getty_df, left_on="getty_index", right_index=True, suffixes=("_museum", "_getty"))
    )
    return merged

In [24]:
chicago_candidates = link_datasets(chicago_artists, artists_clean, threshold=0.85)
chicago_matched_pairs_df = get_matched_pairs(chicago_candidates, chicago_artists, artists_clean)

In [25]:
met_candidates = link_datasets(met_artists, artists_clean, threshold=0.85)
met_matched_pairs_df = get_matched_pairs(met_candidates, met_artists, artists_clean)

In [26]:
# record fusion

def fuse_records(matched_df):
    fused_df = pd.DataFrame()
    
    if "clean_artist" in matched_df.columns:
        fused_df["clean_artist"] = matched_df["clean_artist"]
    elif "clean_artist_museum" in matched_df.columns:
        fused_df["clean_artist"] = matched_df["clean_artist_museum"]
        
    if "name" in matched_df.columns:
        fused_df["artist_name"] = matched_df["name"]
    elif "name_getty" in matched_df.columns:
        fused_df["artist_name"] = matched_df["name_getty"]
        
    if "x" in matched_df.columns:
        fused_df["getty_uri"] = matched_df["x"]
    elif "x_getty" in matched_df.columns:
        fused_df["getty_uri"] = matched_df["x_getty"]
        
    if "bio" in matched_df.columns:
        fused_df["artist_bio"] = matched_df["bio"]
    elif "bio_getty" in matched_df.columns:
        fused_df["artist_bio"] = matched_df["bio_getty"]
        
    if "nationality" in matched_df.columns:
        fused_df["artist_nationality"] = matched_df["nationality"]
    elif "nationality_getty" in matched_df.columns:
        fused_df["artist_nationality"] = matched_df["nationality_getty"]
        
    return fused_df

In [27]:
chicago_fused_df = fuse_records(chicago_matched_pairs_df)
chicago_fused_df.head()

,clean_artist,artist_name,getty_uri,artist_bio,artist_nationality
0,lawrence carmichael earle,"Earle, Lawrence Carmichael",http://vocab.getty.edu/ulan/500029948,"American painter, 1845-1921",American (North American)
1,david dalhoff neal,"Neal, David Dalhoff",http://vocab.getty.edu/ulan/500026278,"American painter, 1838-1915",American (North American)
2,walter shirlaw,"Shirlaw, Walter",http://vocab.getty.edu/ulan/500002781,"American painter and illustrator, 1838-1909",American (North American)
3,salvator rosa,"Rosa, Salvator",http://vocab.getty.edu/ulan/500011328,"Italian painter and engraver, 1615-1673",Italian (culture or style)
4,giovanni battista piranesi,"Piranesi, Giovanni Battista",http://vocab.getty.edu/ulan/500114965,"Italian printmaker, architect, 1720-1778",Italian (culture or style)


In [28]:
met_fused_df = fuse_records(met_matched_pairs_df)
met_fused_df.head()

,clean_artist,artist_name,getty_uri,artist_bio,artist_nationality
0,james barton longacre,"Longacre, James Barton",http://vocab.getty.edu/ulan/500011409,"American printmaker and painter, 1794-1869",American (North American)
1,christian gobrecht,"Gobrecht, Christian",http://vocab.getty.edu/ulan/500077295,"American engraver, 1785-1844",American (North American)
2,christian gobrecht,"Robbrecht, Christian",http://vocab.getty.edu/ulan/500213310,"Belgian architect, contemporary",Belgian (modern)
3,bela lyon pratt,"Pratt, Bela Lyon",http://vocab.getty.edu/ulan/500095555,"American sculptor, 1867-1917",American (North American)
4,pierre stephan,"Stephan, Pierre",http://vocab.getty.edu/ulan/500100483,"French ceramicist, active 1770-1795, active in...",French (culture or style)


In [30]:
chicago_fused_df.to_csv('chicago_fused.csv', index=False)
met_fused_df.to_csv('met_fused.csv', index=False)